Task 1

In [ ]:
import pandas as pd

listings = pd.read_csv('data/listings_clean.csv', parse_dates=['last_scraped', 'calendar_last_scraped', 'first_review', 'last_review'])

# Compare mean and median on a small list before and after adding an extreme value
five = pd.Series([3, 5, 8, 10, 12])
print("Five numbers:", five.mean(), five.median())

six = pd.Series([3, 5, 8, 10, 12, 1293])
print("With 1293:", round(six.mean(), 1), six.median())

print("Reviews mean:", round(listings['number_of_reviews'].mean(), 1))
print("Reviews median:", listings['number_of_reviews'].median())

# We need to use median to answer "how many reviews does a typical listing have?" 
# because half of the listings have 9 reviews or less, 
# while a few super popular places pull the average up to 35.8.

Task 2

In [ ]:
print(listings['number_of_reviews'].describe().round(1))

# Calculate specific percentiles 
print(listings['number_of_reviews'].quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.99]).round(1))

# Calculate q1, q3 and the iqr
q1 = listings['number_of_reviews'].quantile(0.25)
q3 = listings['number_of_reviews'].quantile(0.75)
iqr = q3 - q1

print("Q1:", q1, "Q3:", q3, "IQR:", iqr)

# What the IQR says here: the middle 50% of all listings have between 2 and 29 reviews, so the typical spread is just 27 reviews.
# What the 10th percentile tells: since the 10th percentile is 0, at least 10% of the listings have no reviews at all.

Task 3

In [ ]:
# Calculate lower and upper fences
upper = q3 + 1.5 * iqr
lower = q1 - 1.5 * iqr
print("Lower fence:", lower, "Upper fence:", upper)

# Can there be outliers below the lower fence for this column? 
# No because a listing cannot have a negative number of reviews

# Filter listings above the upper fence and count them
outliers = listings[listings['number_of_reviews'] > upper]
print("Outliers count:", len(outliers))

print("\nRoom types:")
print(outliers['room_type'].value_counts())

print("\nMedian price (outliers):", round(outliers['price'].median(), 1))
print("Median price (others):", round(listings[listings['number_of_reviews'] <= upper]['price'].median(), 1))

# 582 listings are above the upper fence (446 entire homes, 121 private rooms, 14 shared rooms, 1 hotel room). 
# Their median price is 1459.5 compared to 1867.0 for the others, so the listings with the most reviews are cheaper.

Task 4

In [ ]:
# Mark outliers in a new column without removing them, count
listings['reviews_outlier'] = listings['number_of_reviews'] > upper
print("Number of outliers:", listings['reviews_outlier'].sum())

# Build a comparison table for all listings vs without outliers
compare = pd.DataFrame({
    'all': listings['number_of_reviews'].agg(['count', 'mean', 'median', 'std']),
    'without_outliers': listings.loc[~listings['reviews_outlier'], 'number_of_reviews'].agg(['count', 'mean', 'median', 'std'])
}).round(1)

print(compare)

# Calculate percentage change
pct_change = ((compare['without_outliers'] - compare['all']) / compare['all'] * 100).round(0)
print("\nPercentage change:")
print(pct_change)

# Without outliers count drops by 13%, mean by 64%, median by 22%, and std by 80%. 
# Std changes the most and median changes the least.

Task 5